# Introduction

In [1]:
import os
import requests
import string
from collections import Counter

valid = string.ascii_uppercase # creates string of alphabet A-Z
symbols = valid + " "          # adds space since we also count it 

# project_dir = "C:/path/to/project"
project_dir = "."
text_path = f"{project_dir}/moby.txt"
url = "https://www.gutenberg.org/files/2701/2701-0.txt"

# def load_text(path: str, url: str) -> str:
#     if not os.path.exists(path) or os.path.getsize(path) == 0:
#         response = requests.get(url)
#         response.raise_for_status()
#         text = response.content.decode()

#         with open(path, "w", encoding="utf-8") as f:
#             f.write(text)
#     with open(path, "r", encoding="utf-8") as f:
#         return f.read()

if not os.path.exists(text_path) or os.path.getsize(text_path) == 0:
    response = requests.get(url)
    response.raise_for_status()
    text = response.content.decode()

    with open(text_path, "w", encoding="utf-8") as f:
        f.write(text)
with open(text_path, "r", encoding="utf-8") as f:
     moby_text = f.read()

## Permutations

In [2]:
# ANSWER TO QUESTION 1

## Frequency Analysis

We now take a different approach as later ciphers are no longer simple shift ciphers so brute forcing possible permutations is not realistic as there are $27! \approx 1.09 \times 10^{28}$ possible permutions to go through (we include the space).

Instead, by exploiting statistical patterns within the English language, we start by investigating the frequencies of characters in a reference text. In this case we will use the novel Moby Dick written by Herman Melville, and then analyse the cipher before matching the kth most common symbols to decipher.

**The process**:
1. Trim the reference text so that it cuts out licenses, credits, etc.
2. Clean the text by removing all characters that are not in the English alphabet
3. Count the occurrence of each character including spaces.

Note: Steps 1-3 should be carried out for reference texts. For ciphers, step 1 should be skipped.

In [3]:
def text_trimmer(text: str) -> str:
    """
    Input: text - string that you want to be trimmed.
    returns trimmed string so that it only contains proper content.
    NOTE: text must be sourced from gutenberg.org as it searches for its markers
    """

    # error handling
    if not isinstance(text, str):
        raise TypeError(f"Expecting string as a value not {type(text)}")
    if len(text) == 0:
        raise ValueError("Cannot trim an empty string")

    start_marker_idx = text.find("*** START OF")

    # error handling
    if start_marker_idx == -1:
        raise ValueError("Couldn't find '*** START OF' marker - is this a project Gutenberg text?")

    start_idx = text.find("\n", start_marker_idx)
    end_idx = text.find("*** END OF")

    # error handling
    if end_idx == -1 or end_idx <= start_marker_idx:
        raise ValueError("Couldn't find '*** END OF' marker - is this a project Gutenberg text?")

    # slices text excluding the markers
    return text[start_idx:end_idx]


def text_cleaner(text: str) -> str:
    """
    Input: text - string that you want to clean.
    returns capitalised string including only letters in A-Z and space
    """
    # error handling
    if not isinstance(text, str):
        raise TypeError(f"Expecting string as a value not {type(text)}")
    if len(text) == 0:
        raise ValueError("Cannot clean an empty string")

    # checks if each character is in A-Z, if not replace with space.
    cleaned = "".join(
        letter if letter in valid else " " for letter in text.upper())

    # collapses all whitespace into a single space
    cleaned = " ".join(cleaned.split())

    return cleaned


def text_per(text: str, percentage: bool = True) -> list:
    """
    Input: text - string that you want the count/ percentage to. Percentage - boolean. If True (default) returns percentage, else returns counts.
    text_cleaner should be ran before this.
    """

    # error handling
    if not isinstance(text, str):
        raise TypeError(f"Expecting string as a value not {type(text)}")
    if not isinstance(percentage, bool):
        raise TypeError(f"Expecting boolean as a value not {type(percentage)}")
    if len(text) == 0:
        raise ValueError("Cannot count an empty string")
    if len(set(text) - set(symbols)) != 0:
        raise ValueError("text contains characters other than A-Z and space. Run text_cleaner first")

    total_count = len(text) # length of total text
    counts = Counter(text)  # counts occurrence of each character

    # creates a dictionary with characters and their counts
    text_count = {symbol: counts[symbol] for symbol in symbols} 

    # sort character by count
    by_text_count = sorted(
        text_count.items(), key=lambda pair: pair[1], reverse=True)

    # character by percentage
    text_per = {
        char: (count/total_count) * 100 for char, count in text_count.items()
    }

    # sort chacter by percentage
    by_text_per = sorted(
        text_per.items(), key=lambda pair: pair[1], reverse=True)

    if percentage:
        return by_text_per
    else:
        return by_text_count

## Question 3

## Question 4

## Question 5

In Question 4, we explored how manually swapping character mappings can improve the readability of a decrypted message. However, visual inspection alone is not always sufficient. Therefore, in this question, we develop a bigram probability model to assign a numerical score to potential decoded messages.

### 5.1 Data Preparation

We use the provided reference text, *Moby-Dick*, to estimate character and bigram frequencies. Before counting, the text is processed using the provided `text_trimmer` and `text_cleaner` functions to ensure that it follows the required character set.

In [4]:
import math
from collections import Counter

reference_text = text_cleaner(text_trimmer(moby_text))

print("Reference text prepared successfully.")
print("Total characters:", len(reference_text))

Reference text prepared successfully.
Total characters: 1174222


The processed reference text contains 1,174,222 characters. This text is used as the basis for all subsequent frequency calculations.

### 5.2 Character and Bigram Frequencies

A bigram is a pair of consecutive characters. Since our character set contains 27 symbols, there are $$27^2 = 729$$ possible bigrams.

We use Python's `Counter` to count the occurrences of individual characters and consecutive character pairs in the reference text.

In [5]:
# Count the frequency of individual characters
char_counts = Counter(reference_text)

# Count the frequency of consecutive character pairs
bigram_counts = Counter(
    zip(reference_text[:-1], reference_text[1:])
)

# Create all 729 possible bigrams
all_bigram_counts = {
    (i, j): bigram_counts.get((i, j), 0)
    for i in symbols
    for j in symbols
}

print("Number of characters:", len(char_counts))
print("Number of possible bigrams:", len(all_bigram_counts))
print("Total bigram occurrences:", sum(bigram_counts.values()))

print("Frequency of H:", char_counts["H"])
print("Frequency of HE:", bigram_counts[("H", "E")])
print("Frenquency of TH:", bigram_counts[("T", "H")])

Number of characters: 27
Number of possible bigrams: 729
Total bigram occurrences: 1174221
Frequency of H: 63237
Frequency of HE: 26631
Frenquency of TH: 31492


The reference text contains 1,174,221 bigram occurrences. This is consistent with the expected number, since a text containing $N$ characters has $N − 1$ consecutive character pairs: $$1,174,222 - 1 = 1,174,221$$

The frequencies of selected characters and bigrams, such as H, HE and TH, provide examples of the counting results.

### 5.3 Bigram Probability

For each pair of characters $i$ and $j$, we calculate its probability using the formula specified in the project:

$$
p(i,j)=
\frac{\operatorname{frequency}(ij)+1}
{\operatorname{frequency}(i)}.
$$

Here, $\operatorname{frequency}(ij)$ is the number of occurrences of the bigram $ij$, while $\operatorname{frequency}(i)$ is the number of occurrences of character $i$.

The additional 1 acts as a fudge factor, ensuring that unseen bigrams are assigned a non-zero probability. We calculate and store the probabilities of all 729 possible bigrams in a dictionary.

In [6]:
# Calculate the probability of a bigram
def bigram_probability(i, j):
    numerator = bigram_counts[(i, j)] + 1
    denominator = char_counts[i]

    if denominator == 0:
        raise ValueError(f"Character {i!r} does not occur in reference text.")

    return numerator / denominator

# Calculate probabilities for all 729 possible bigrams
bigram_probabilities = {
    (i, j): bigram_probability(i, j)
    for i in symbols
    for j in symbols
}

# Test some probabilities
print("P(H, E) =", bigram_probability("H", "E"))
print("P(T, H) =", bigram_probability("T", "H"))
print("P(Q, Z) =", bigram_probability("Q", "Z"))

P(H, E) = 0.4211458481585148
P(T, H) = 0.3563241799893645
P(Q, Z) = 0.0006369426751592356


### 5.4 Message Scoring

For a potential decoded message

$$
m=i_1i_2\ldots i_n,
$$

the project defines its score as:

$$
S(m)=\sum_{j=1}^{n-1}\log p(i_j,i_{j+1}).
$$

The function adds the logarithms of the probabilities of all consecutive character pairs. Using logarithms converts a product of probabilities into a sum, making the calculation more convenient.

A higher score indicates a more plausible message under this bigram model, particularly when comparing messages of similar lengths.

In [7]:
# Calculate the score of a message
def score_message(message):
    # Check input type
    if not isinstance(message, str):
        raise TypeError("Message must be a string.")

    # Handle an empty message
    if message == "":
        return 0.0

    # Clean the message
    message = text_cleaner(message)

    # Initialise the score
    score = 0.0

    # Go through every consecutive pair of characters
    for i, j in zip(message[:-1], message[1:]):

        # Add the logarithm of the bigram probability
        score += math.log(bigram_probabilities[(i, j)])

    return score

# Test the scoring function
print("Score of THE:", score_message("THE"))
print("Score of HELLO WORLD:", score_message("HELLO WORLD"))
print("Score of THIS IS AN ENGLISH MESSAGE:",
     score_message("THIS IS AN ENGLISH MESSAGE"))

Score of THE: -1.8966904171565897
Score of HELLO WORLD: -25.574543315535138
Score of THIS IS AN ENGLISH MESSAGE: -58.027163202805795


Since bigram probabilities are generally between zero and one, their logarithms are usually negative. Therefore, this explains why the resulting message scores are typically negative.

### 5.5 Testing and Results

We test the scoring function using messages of different lengths, including ordinary English text, an empty message and a single-character message.

These tests help verify that the function calculates scores correctly and handles messages containing no consecutive character pairs.

In [8]:
# Test special cases
print("Empty message:", score_message(""))
print("Single character:", score_message("A"))

# Test a longer message
test_message = "THE QUICK BROWN FOX"
print("Test message:", test_message)
print("Message score:", score_message(test_message))

Empty message: 0.0
Single character: 0.0
Test message: THE QUICK BROWN FOX
Message score: -47.066818975249284


The empty message and single-character message both receive a score of zero, as they contain no consecutive character pairs. These test results demonstrate that the scoring function works for both ordinary messages and these edge cases.

In contrast, the longer test message receives a negative score. This happens because it contains many character pairs with probabilities less than one, and it illustrates how the function combines the probabilities of consecutive character pairs into a single score.

In [9]:
# Test an unseen bigram
unseen_pair = next(
    (i, j)
    for i in symbols
    for j in symbols
    if bigram_counts[(i, j)] == 0
)

print("Unseen bigram:", unseen_pair)
print("Bigram frequency:", bigram_counts[unseen_pair])
print("Bigram probability:", bigram_probability(*unseen_pair))

assert bigram_probability(*unseen_pair) > 0

print("Test passed: unseen bigrams have non-zero probabilities.")

Unseen bigram: ('B', 'C')
Bigram frequency: 0
Bigram probability: 5.9059768485707536e-05
Test passed: unseen bigrams have non-zero probabilities.


We also test a unseen bigram that does not occur in the reference text. Its probability remains positive due to the additional 1 in the numerator, confirming that the method avoids zero probabilities.

### 5.6 Discussion and Limitations

The bigram model simply estimates the plausibility of a message by the frequency of character pairs.

However, this score depends on the length of the message because it is derived by summing multiple logarithmic probabilities. Therefore, when comparing messages of different lengths, the scores should be treated with caution.

Furthermore, this model only considers the pairing of consecutive characters and cannot capture more complex linguistic structures, such as complete words or grammar.

Therefore, even if a message does not form a meaningful English sentence, it may still receive a relatively high score due to the presence of common character pairs.

The scoring function developed in this question will be used in Question 6, where the Metropolis algorithm uses the score difference to guide the search and find a better decoded message.

## Question 6